In [9]:
import pandas as pd
import folium

In [10]:
data = pd.read_csv("tn_nashville_2020_04_01.csv")
location_counts = data['location'].value_counts()[1:]
popular_locations = location_counts[location_counts >= 1500]
filtered_data = data[data['location'].isin(popular_locations.index)].reset_index()

C:\Users\partu\AppData\Local\Temp\ipykernel_17588\1120002060.py:1: DtypeWarning: Columns (6,8,15,16,17,22,23,24,25,29,30,31,32,33,35,36,37,38,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("tn_nashville_2020_04_01.csv")


In [11]:
popular_locations

location
BRILEY PKWY N & MURFREESBORO PIKE, NASHVILLE, TN, 37217      21759
HARDING PL & NOLENSVILLE PIKE, NASHVILLE, TN, 37211          14981
BRILEY PKWY E & GALLATIN PIKE, NASHVILLE, TN, 37216          10388
BRILEY PKWY N & MCGAVOCK PIKE, NASHVILLE, TN, 37214           9990
NOLENSVILLE PIKE & OLD HICKORY BLVD, NASHVILLE, TN, 37211     8785
                                                             ...  
BRILEY PKWY & MCGAVOCK PIKE, NASHVILLE, TN, 37214             1510
12TH AVE S & ARCHER ST, NASHVILLE, TN, 37203                  1505
5200 FRANKLIN PIKE, NASHVILLE, TN, 37220                      1504
40TH AVE N & CLIFTON AVE, NASHVILLE, TN, 37209                1503
14TH AVE N & BROADWAY, NASHVILLE, TN, 37203                   1500
Name: count, Length: 311, dtype: int64

In [12]:
sex_counts = filtered_data.groupby(['location', 'subject_sex']).size().reset_index(name='count')
total_counts = filtered_data.groupby('location').size().reset_index(name='total_count')
sex_counts = sex_counts.merge(total_counts, on='location')
sex_counts['percentage'] = (sex_counts['count'] / sex_counts['total_count'] * 100).round(2)
pivot_table = sex_counts.pivot_table(index='location', columns='subject_sex', values='percentage', fill_value=0).reset_index()
pivot_table.columns.name = None
pivot_table.columns = ['location', 'female', 'male']
pivot_table = pivot_table[['location', 'male', 'female']]
pivot_table.head()

,location,male,female
0,"1033 ANTIOCH PIKE, NASHVILLE, TN, 37211",61.91,38.09
1,"1084 TULIP GROVE RD, HERMITAGE, TN, 37076",49.74,49.80
2,"10TH AVE N & BROADWAY, NASHVILLE, TN, 37203",65.04,33.99
3,"10TH AVE S & BROADWAY, NASHVILLE, TN, 37203",64.56,35.40
4,"1101 BROADWAY, NASHVILLE, TN, 37203",69.08,30.67


In [ ]:
filtered_data = filtered_data.dropna(subset=['lat', 'lng'])
unique_locations = filtered_data.groupby(['lat', 'lng']).first().reset_index()

m = folium.Map(location=[unique_locations['lat'].mean(), unique_locations['lng'].mean()], zoom_start=12)

for index, row in unique_locations.iterrows():
    folium.Marker(
        [row['lat'], row['lng']], 
        popup=f"{row['date']} - {row['violation']} - {row['outcome']}"
    ).add_to(m)

m.save('nashville_map.html')

In [14]:
from IPython.display import IFrame

IFrame('nashville_map.html', width=800, height=600)